# sum-back-expand-broadcast — ex2: multi-axis sum_back — collapse two axes, restore both via unsqueeze chain

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sum-back-expand-broadcast`. Running the final beacon cell reports progress against the `Backprop: sum_back via expand_broadcast` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: sum_back via expand_broadcast` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sum-back-expand-broadcast`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sum-back-expand-broadcast"
DD_SUBTOPIC = "Backprop: sum_back via expand_broadcast"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Multi-axis `sum_back`: collapsing TWO axes — quick refresher

ex1 collapsed a single dim. The deeper facet is `dim=(d0, d1)` — a multi-axis sum. The expand-broadcast pattern still works, but you must restore EACH collapsed axis as size 1 before expanding.

```
x.shape       = (2, 3, 4, 5)
out = x.sum(dim=(0, 2))           # out.shape = (3, 5)
grad_out.shape = (3, 5)

# restore BOTH collapsed axes as size-1 (sorted ascending so indexing stays valid):
g = grad_out.unsqueeze(0).unsqueeze(2)   # (1, 3, 1, 5)
grad_in = g.expand(x.shape)              # (2, 3, 4, 5)
```

Why sort the dims ascending. `unsqueeze(0)` shifts every later axis by +1 — so if you insert at axis 0 first, axis 2 IN THE OUTPUT corresponds to axis 2 in the FINAL tensor (the next insertion target). Reverse order (insert at 2 first, then 0) also works but the bookkeeping is trickier — ascending is the standard convention.

### Exercise 2 — multi-axis sum_back — collapse two axes, restore both via unsqueeze chain

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply expand-broadcast to multi-axis sum: restore EACH collapsed axis as size-1 via successive unsqueeze, then expand to x.shape.
> Keywords: sum, multi-axis, expand, unsqueeze, broadcast
> ```

**KCs targeted:** `sum-backward-pattern`, `kwargs-pass-through-recipe`

Implement `sum_back_multi(grad_out, out, x, dims, keepdim=False)` for the forward op `out = x.sum(dim=dims, keepdim=keepdim)` where `dims` is a TUPLE of ints (not a single int).

Derivation:
- Each element of `x` contributes to exactly ONE output entry.
- Backward broadcasts `grad_out` back along EVERY collapsed axis.

**`keepdim=False`** — `grad_out` is missing every dim in `dims`. Restore them as size-1 axes one at a time, then expand:
```
g = grad_out
for d in sorted(dims):            # ascending so inserts don't shift each other
    g = g.unsqueeze(d)
grad_in = g.expand(x.shape)
```

**`keepdim=True`** — `grad_out` already has size-1 at every `d in dims`. Just `expand` directly:
```
grad_in = grad_out.expand(x.shape)
```

**Why sort ascending.** `unsqueeze(d)` shifts later axis positions by +1 — but only ones LATER than `d`. If you process dims in ascending order, each insertion happens at the same numerical index it had in the original `dims` tuple (because earlier insertions only added axes BEFORE it, leaving its index intact AS the next axis to insert at). Descending order also works but the bookkeeping inverts.

Return a `torch.Tensor` with `x.shape`. No autograd.

In [ ]:
def sum_back_multi(grad_out: Tensor, out: Tensor, x: Tensor, dims: tuple, keepdim: bool = False) -> Tensor:
    if keepdim:
        return grad_out.expand(x.shape)
    g = grad_out
    # Ascending order: each unsqueeze leaves later indices in dims intact.
    for d in sorted(dims):
        g = g.unsqueeze(d)
    return g.expand(x.shape)


<details><summary>Solution</summary>

```python
def sum_back_multi(grad_out: Tensor, out: Tensor, x: Tensor, dims: tuple, keepdim: bool = False) -> Tensor:
    if keepdim:
        return grad_out.expand(x.shape)
    g = grad_out
    # Ascending order: each unsqueeze leaves later indices in dims intact.
    for d in sorted(dims):
        g = g.unsqueeze(d)
    return g.expand(x.shape)
```

**Why ascending order for unsqueeze.** Suppose `dims=(0, 2)`. After `unsqueeze(0)`, the tensor gains a leading axis: indices 1, 2, ... in the result correspond to indices 0, 1, ... in the input. Axis `2` in the FINAL output is still axis `2` in the current state — because the new axis was inserted BEFORE it. Descending order would need index correction.

**Why `sorted(dims)` even if the caller already sorted.** Callers shouldn't be obliged to sort — and dispatchers don't reorder Recipe kwargs. Internal `sorted()` keeps the back fn robust to whatever the forward wrapper stored.

**Where multi-axis sum_back shows up.** Layer-norm reduces over the last `K` axes simultaneously. Cross-entropy over `(batch, classes)` reduces over classes. Any reduction with `dim=tuple` uses this exact back fn — and getting the unsqueeze order wrong is a classic bug.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()